# Fed-SALGATE — Part 2 · Novel Model + Centralized Baseline (FSCG-UNet)

**Federated Saliency-Consensus Gating for Drone-Based Flood Damage Assessment**

Part 2 builds the **new segmentation model** behind FSCG and trains it **centrally**
as the strong baseline, using only the real FloodNet GT prepared by Part 1.

**Novel model — FSCG-UNet** (Federated Saliency-Consensus Gated U-Net):
- **Hybrid CNN–ViT encoder** (`ConvNeXt-Tiny` features + patch-embed ViT transformer
  bridge for long-range flood-boundary context) — the architecture direction of the
  reference `hybrid-cnn-vit-attention-u-net`.
- **ASPP** multi-scale context aggregation.
- **Attention-gated skip fusion** in the decoder (guided by gating signal).
- **Spatial Saliency-Consensus Gates (SSCG)**: each decoder stage emits a spatial
  saliency mask that re-weights its features *and* is returned as an interpretable
  saliency map — the in-model counterpart of the server-side FSCG gate (later used by
  Part 3 for consensus scoring, replacing costly Grad-CAM).
- **Deep supervision** (main + aux2 + aux3) with a composite loss
  (weighted CE + Dice + Focal + Lovász).
- **Training strategy** (from `Training optimization strategies.md`): cuDNN autotuner,
  `pin_memory` + `persistent_workers` loaders, GPU-vectorized mIoU, automatic mixed
  precision (`autocast` + `GradScaler`), TTA at eval, cosine annealing warm restarts,
  gradient clipping, early stopping.

**Inputs:** `fed_salgate_outputs` from Part 1 (config.json, trainer_samples.json, val/test samples, class weights).
**Outputs (reused by Part 3):** best checkpoint, `model_spec.json`, metrics JSON, plots.

Run on Kaggle free T4 in the background. Attach Part 1's Output as a data source.

## ⚙️ Install packages

In [ ]:
import subprocess, sys
import glob

packages = [
    "segmentation-models-pytorch==0.3.3",
    "albumentations==1.3.1",
    "timm",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Packages ready.")

## 📚 Imports

In [ ]:
import os, sys, json, random, warnings, time, pickle, copy, math
from pathlib import Path
from collections import defaultdict
from itertools import filterfalse

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image

import timm

warnings.filterwarnings("ignore")
print("Imports OK. PyTorch", torch.__version__, "| timm", timm.__version__)

## ⚙️ Config + Part-1 discovery (any device)

In [ ]:
SMOKE_TEST = os.environ.get("FSCG_SMOKE") == "1"
DEBUG_SHORT = os.environ.get("FSCG_DEBUG") == "1" or SMOKE_TEST

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IN_KAGGLE = os.path.isdir("/kaggle/input")
if IN_KAGGLE:
    WORKING_DIR = "/kaggle/working"
else:
    WORKING_DIR = os.path.join(os.path.dirname(os.getcwd()), "working")
    os.makedirs(WORKING_DIR, exist_ok=True)

OUTPUT_DIR = os.path.join(WORKING_DIR, "fed_salgate_outputs")
CKPT_DIR   = os.path.join(WORKING_DIR, "checkpoints")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
for d in [OUTPUT_DIR, CKPT_DIR, PLOT_DIR, os.path.join(OUTPUT_DIR, "tables")]:
    os.makedirs(d, exist_ok=True)


def _find_part1_dir():
    """Locate Part-1 outputs. Priority:
    1. env PART1_DIR (already handled by caller),
    2. /kaggle/working (inline run in the same session),
    3. the Kaggle "Output/Model" import path (Part-1 output attached as an input):
       /kaggle/input/models/<user>/fed-salgate-outputs/tensorflow2/default/1/fed_salgate_outputs
    4. shallow scan of /kaggle/input (custom input datasets version).
    """
    if os.path.exists(os.path.join(OUTPUT_DIR, "config.json")):
        return OUTPUT_DIR
    if IN_KAGGLE and os.path.exists("/kaggle/input/models/YOUR_KAGGLE_USERNAME/feds1/tensorflow2/default/1/fed_salgate_outputs" + os.sep + "config.json"):
        return "/kaggle/input/models/YOUR_KAGGLE_USERNAME/feds1/tensorflow2/default/1/fed_salgate_outputs"
    if IN_KAGGLE:
        # 2.5) explicit known output slugs (edit/add to match THIS notebook's saved Part-1 Output)
        for _slug in ("feds1", "fed-salgate-output", "fed-salgate-outputs", "fed-salgate"):
            _cand = os.path.join("/kaggle/input", "models", "*", _slug,
                                "tensorflow2", "default", "1", "fed_salgate_outputs")
            for _m in glob.glob(_cand):
                if os.path.exists(os.path.join(_m, "config.json")):
                    return _m
    if IN_KAGGLE:
        # 3) exact "models" import path first (avoids scanning huge dataset trees)
        # wildcard the owner AND slug: Kaggle output slug may be fed-salgate-output or fed-salgate-outputs
        for cand in glob.glob(os.path.join("/kaggle/input", "models", "*", "*",
                                          "tensorflow2", "default", "1")):
            cand = os.path.join(cand, "fed_salgate_outputs")
            if os.path.exists(os.path.join(cand, "config.json")):
                return cand
        # also try fed_salgate_output directly under any model dir (varying depth)
        for cand in glob.glob(os.path.join("/kaggle/input", "models", "*", "*", "fed_salgate_outputs")):
            if os.path.exists(os.path.join(cand, "config.json")):
                return cand
        # 4) shallow scan, skip the giant raw dataset dirs
        for r in ["/kaggle/input", "/kaggle/input/datasets"]:
            for d in sorted(os.listdir(r)) if os.path.isdir(r) else []:
                cand = os.path.join(r, d, "fed_salgate_outputs")
                if os.path.exists(os.path.join(cand, "config.json")):
                    return cand
    return None


PART1_DIR = os.environ.get("PART1_DIR", "") or (_find_part1_dir() or "")
print(f"Part 1 outputs: {PART1_DIR}")
if not PART1_DIR or not os.path.exists(os.path.join(PART1_DIR, "config.json")):
    print("WARNING: Part-1 outputs not found -> using Part-1 fallback defaults.")

with open(os.path.join(PART1_DIR, "config.json")) as f:
    CFG = json.load(f)

NUM_CLASSES = int(CFG.get("NUM_CLASSES", 10))
CLASS_NAMES = CFG.get("CLASS_NAMES", ['Background', 'Building-Flooded', 'Building-NonFlooded',
                                      'Road-Flooded', 'Road-NonFlooded', 'Water',
                                      'Tree', 'Vehicle', 'Pool', 'Grass'])
IMG_SIZE     = int(os.environ.get("FSCG_IMG", "64" if SMOKE_TEST else 320))
BATCH_SIZE   = int(os.environ.get("FSCG_BATCH", 4)) if not SMOKE_TEST else 2
class_weights = CFG.get("class_weights")
NUM_CLIENTS   = int(CFG.get("NUM_CLIENTS", 3))

print(f"IMG_SIZE={IMG_SIZE}  BATCH={BATCH_SIZE}  NUM_CLASSES={NUM_CLASSES}")
print(f"CLASS_NAMES={CLASS_NAMES}")

# ---- runtime knobs (safe defaults; oversampled 2560-set is STILL the train data) ----
FSCG_EPOCHS     = os.environ.get("FSCG_EPOCHS", "3" if SMOKE_TEST else "100")
FSCG_MAX_HOURS  = os.environ.get("FSCG_MAX_HOURS", "8.0")
FSCG_VALLOG     = os.environ.get("FSCG_VALLOG", "64")
FSCG_WORKERS    = os.environ.get("FSCG_WORKERS", "4")
FSCG_VAL_EVERY  = os.environ.get("FSCG_VAL_EVERY", "1")
EPOCHS          = int(FSCG_EPOCHS)
MAX_HOURS       = float(FSCG_MAX_HOURS)
VAL_LOG_N       = int(FSCG_VALLOG)
VAL_EVERY       = max(1, int(FSCG_VAL_EVERY))
WORKERS         = int(FSCG_WORKERS)
print(f"Runtime: EPOCHS={EPOCHS} MAX_HOURS={MAX_HOURS}h VAL_LOG_N={VAL_LOG_N} "
      f"VAL_EVERY={VAL_EVERY} WORKERS={WORKERS}")
print(">>> FAST-320 v3: 320px | batch 4 | compile=OFF | 8h budget | pre-resize cache (fixes flat per-epoch time from full-res decode)")


## 🧬 Dataset rebuild (identical order to Part 1)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- normal augmentation for originals (Part-1 aug_normal equivalent) ---
aug_train = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-30, 30), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.RandomGamma(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

# --- strong augmentation for oversampled copies (Part-1 aug_strong equivalent) ---
aug_strong = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.8, 1.2), rotate=(-45, 45), p=0.7),
    A.OneOf([A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.5),
             A.GridDistortion(p=0.5)], p=0.4),
    A.OneOf([A.RandomBrightnessContrast(p=0.5),
             A.HueSaturationValue(p=0.5),
             A.CLAHE(p=0.5)], p=0.5),
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

aug_eval = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)


class FloodSegDataset(Dataset):
    """Original images use aug_normal; oversampled copies use aug_strong.
    flags is None for val/test (deterministic aug_eval)."""
    def __init__(self, samples, transform, transform_strong=None, flags=None):
        self.samples = samples
        self.transform = transform
        self.transform_strong = transform_strong
        self.flags = flags

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img  = cv2.imread(s["img"]);           img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE)
        if img.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        if self.flags is not None and self.transform_strong is not None and self.flags[idx]:
            transform = self.transform_strong
        else:
            transform = self.transform
        aug = transform(image=img, mask=mask)
        return aug["image"], aug["mask"].long(), os.path.basename(s["img"])


# --- load (oversampled) train samples + copy flags from Part 1 ---
_ov_path = os.path.join(PART1_DIR, "train_samples_oversampled.json")
_fl_path = os.path.join(PART1_DIR, "oversample_flags.json")
if os.path.exists(_ov_path) and os.path.exists(_fl_path):
    with open(_ov_path) as f:
        train_samples = json.load(f)
    with open(_fl_path) as f:
        oversample_flags = json.load(f)
    print(f"Loaded oversampled train set ({len(train_samples)} samples, "
          f"{int(sum(oversample_flags))} copies).")
else:
    with open(os.path.join(PART1_DIR, "train_samples.json")) as f:
        train_samples = json.load(f)
    oversample_flags = [False] * len(train_samples)
    print("Oversampled files not found -> using originals only (no copies).")

with open(os.path.join(PART1_DIR, "val_samples.json")) as f:
    val_samples = json.load(f)
with open(os.path.join(PART1_DIR, "test_samples.json")) as f:
    test_samples = json.load(f)

# --- remap Part-1 absolute paths to THIS session's dataset mounts ---
# Part 1 recorded full paths (e.g. under /kaggle/input/datasets/...). The same
# dataset can be mounted under a different slug in the Part-2 notebook, so we
# re-point img/mask paths at the roots that actually exist here.
def _find_dataset_roots():
    roots = {}
    fb = os.environ.get("FLOODNET_BASE", "")
    sb = os.environ.get("SECONDARY_BASE", "")
    if fb and os.path.isdir(fb):
        roots["floodnet"] = fb
    if sb and os.path.isdir(sb):
        roots["secondary"] = sb
    if IN_KAGGLE and (not roots.get("floodnet") or not roots.get("secondary")):
        for d in os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []:
            base = os.path.join("/kaggle/input", d)
            if "floodnet" not in roots:
                cand = os.path.join(base, "FloodNet Challenge - Track 1")
                if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "Train", "Labeled")):
                    roots["floodnet"] = cand
            if "floodnet" not in roots:
                for sub in os.listdir(base) if os.path.isdir(base) else []:
                    cand = os.path.join(base, sub, "FloodNet Challenge - Track 1")
                    if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "Train", "Labeled")):
                        roots["floodnet"] = cand
            if "secondary" not in roots:
                cand = os.path.join(base, "floodnet_fixed")
                if os.path.isdir(os.path.join(cand, "data_finally")):
                    roots["secondary"] = cand
            if "secondary" not in roots:
                for sub in os.listdir(base) if os.path.isdir(base) else []:
                    cand = os.path.join(base, sub, "floodnet_fixed")
                    if os.path.isdir(os.path.join(cand, "data_finally")):
                        roots["secondary"] = cand
    return roots

def _remap_path(p, roots):
    if os.path.exists(p):
        return p
    for marker, key in [("/data_finally/", "secondary"),
                        ("/Train/Labeled/", "floodnet"),
                        ("FloodNet Challenge - Track 1", "floodnet")]:
        i = p.find(marker)
        if i != -1 and roots.get(key):
            cand = os.path.join(roots[key], p[i + len(marker):].lstrip("/"))
            if os.path.exists(cand):
                return cand
    return p

_ds_roots = _find_dataset_roots()
_n_remap = 0
for _samples in (train_samples, val_samples, test_samples):
    for _s in _samples:
        for _k in ("img", "mask"):
            _new = _remap_path(_s[_k], _ds_roots)
            if _new != _s[_k]:
                _s[_k] = _new
                _n_remap += 1
print(f"Remapped {_n_remap} Part-1 sample paths to this session's dataset mount "
      f"(floodnet={_ds_roots.get('floodnet')} secondary={_ds_roots.get('secondary')})")

# ---- FAST-320 v3: pre-resize cache (kills the full-res CPU decode floor) ----
# Per-epoch time was FLAT across IMG_SIZE (320/384/512 ~ same) -> the wall was
# cv2.imread at full original resolution + resize per __getitem__, not GPU pixel
# work. Resize every image once now; loaders then read small cached files.
try:
    if os.environ.get("FSCG_CACHE", "1") == "1":
        _cache_dir = os.path.join(WORKING_DIR, "resized_cache", str(IMG_SIZE))
        os.makedirs(_cache_dir, exist_ok=True)

        def _resize_to_cache(s, split):
            img = cv2.imread(s["img"])
            msk = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE)
            if img is None or msk is None:
                return s
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
            stem = os.path.splitext(os.path.basename(s["img"]))[0]
            _ip = os.path.join(_cache_dir, f"{split}_{stem}.jpg")
            _mp = os.path.join(_cache_dir, f"{split}_{stem}_mask.png")
            cv2.imwrite(_ip, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            cv2.imwrite(_mp, msk)
            return {"img": _ip, "mask": _mp}

        _t0c = time.time()
        import concurrent.futures as _cf
        _tasks = ([(s, "tr") for s in train_samples] +
                  [(s, "va") for s in val_samples] +
                  [(s, "te") for s in test_samples])
        _nw = max(1, min(4, os.cpu_count() or 1))
        with _cf.ProcessPoolExecutor(max_workers=_nw) as _ex:
            _out = list(_ex.map(_resize_to_cache,
                                [t[0] for t in _tasks], [t[1] for t in _tasks]))
        _ntr = len(train_samples)
        train_samples = _out[:_ntr]
        val_samples   = _out[_ntr:_ntr + len(val_samples)]
        test_samples  = _out[_ntr + len(val_samples):]
        print(f"Pre-resize cache ready: {len(_tasks)} samples -> {_cache_dir} "
              f"({time.time()-_t0c:.0f}s, {_nw} procs); loaders now read small cached files")
    else:
        print("FSCG_CACHE=0 -> reading full-res originals (slow path)")
except Exception as _ec:
    print("Pre-resize cache FAILED -> continuing with full-res originals:", _ec)


train_dataset = FloodSegDataset(train_samples, aug_train, aug_strong, oversample_flags)
val_dataset   = FloodSegDataset(val_samples,   aug_eval)
test_dataset  = FloodSegDataset(test_samples,  aug_eval)
print(f"train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")


## 🛡️ Leak audit — disjoint splits, deterministic validation, label-safe TTA

In [ ]:
leak_issues = []

# 1) dataset path-disjointness (samples come from Part-1 JSON lists)
tr_paths = {s["img"] for s in train_samples}
va_paths = {s["img"] for s in val_samples}
te_paths = {s["img"] for s in test_samples}
for tag, A, B in [("train+val", tr_paths, va_paths), ("train+test", tr_paths, te_paths),
                  ("val+test", va_paths, te_paths)]:
    if A & B:
        leak_issues.append(f"path overlap in {tag}: {len(A & B)}")

# 2) transform separation: val/test must be deterministic (Resize+Normalize only)
def _leaf_names(t):
    out = []
    for a in getattr(t, "transforms", []):
        if hasattr(a, "transforms"):
            out += _leaf_names(a)
        else:
            out.append(type(a).__name__)
    return out or [type(t).__name__]

_eval_names  = _leaf_names(aug_eval)
_train_names = _leaf_names(aug_train)
_strong_names = _leaf_names(aug_strong)
if not (_eval_names == ["Resize", "Normalize", "ToTensorV2"]):
    leak_issues.append(f"aug_eval is not deterministic: {_eval_names}")
_stochastic = {"HorizontalFlip", "VerticalFlip", "RandomRotate90", "Affine",
               "RandomBrightnessContrast", "ElasticTransform", "GridDistortion",
               "RandomGamma"}
# stochastic ops may live in EITHER the normal or strong train pipeline (copies use strong)
_train_union = set(_train_names) | set(_strong_names)
if not _stochastic.issubset(_train_union):
    leak_issues.append(f"train pipelines missing stochastic ops: {_stochastic - _train_union}")

# 3) object identity: train/val/test use their intended pipelines
if train_dataset.transform is not aug_train:
    leak_issues.append("train loader not using aug_train")
if val_dataset.transform is not aug_eval or test_dataset.transform is not aug_eval:
    leak_issues.append("val/test loader not using deterministic aug_eval")

if leak_issues:
    for x in leak_issues:
        print("!! LEAK:", x)
    raise AssertionError(f"Leak audit FAILED: {leak_issues}")
print("Leak audit PASSED: splits disjoint | train=stochastic | val/test=Resize+Normalize "
      "| TTA is deterministic flip-averaging and never touches labels.")

# # Online class-balanced sampling: only needed if Part-1 oversampled set was NOT used.
if os.path.exists(os.path.join(PART1_DIR, "train_samples_oversampled.json")):
    sampler = None
    shuffle_train = True
    print("Using oversampled train set with plain shuffling (already class-balanced).")
elif class_weights is not None and len(class_weights) == NUM_CLASSES:
    weights = []
    for s in train_samples:
        try:
            mk = np.array(Image.open(s["mask"]))
        except Exception:
            weights.append(1.0); continue
        if mk.ndim == 3:
            mk = mk[:, :, 0]
        present = np.unique(mk).astype(int)
        w = np.max([class_weights[c] for c in present]) if len(present) else 1.0
        weights.append(float(w))
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    shuffle_train = False
    print("Fallback: class-weighted online sampling (no oversampled set found).")
else:
    sampler = None
    shuffle_train = True
    print("Using plain shuffling (no class weights found).")

In [ ]:
# ---- per-class pixel presence diagnostic (why some classes have near-zero IoU) ----
def _pixel_counts(samples):
    counts = np.zeros(NUM_CLASSES)
    for s in samples:
        mk = np.array(Image.open(s["mask"]))
        if mk.ndim == 3: mk = mk[:, :, 0]
        counts += np.bincount(mk.ravel(), minlength=NUM_CLASSES)[:NUM_CLASSES]
    return counts

def _present(samples):
    out = np.zeros(NUM_CLASSES)
    for s in samples:
        mk = np.array(Image.open(s["mask"]))
        if mk.ndim == 3: mk = mk[:, :, 0]
        for _u in np.unique(mk):
            if 0 <= _u < NUM_CLASSES:
                out[int(_u)] = 1
    return out

_diag = pd.DataFrame({
    "val_pixels":  _pixel_counts(val_samples),
    "test_pixels": _pixel_counts(test_samples),
    "val_present": _present(val_samples),
    "test_present": _present(test_samples),
}, index=[f"{c} {CLASS_NAMES[c]}" for c in range(NUM_CLASSES)])
print("Per-class pixel presence (val/test): classes with 0 pixels never appear -> IoU=0 is a data artifact, not a model failure.")
print(_diag.to_string())
_absent = [(CLASS_NAMES[c], "val" if _diag["val_pixels"].iloc[c]==0 else "") for c in range(NUM_CLASSES)]
with open(os.path.join(OUTPUT_DIR, "tables", "p2_class_presence.csv"), "w") as f:
    _diag.to_csv(f)
print("Saved tables/p2_class_presence.csv")

In [ ]:
from PIL import Image as PILImage
Image = PILImage

# cuDNN autotuner (Optimization 1)
HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
if HAS_CUDA:
    torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)

num_workers = WORKERS if (not SMOKE_TEST and HAS_CUDA) else 0
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=shuffle_train,
                          sampler=sampler, num_workers=num_workers, pin_memory=True,
                          persistent_workers=(num_workers > 0), drop_last=True)
val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=True,
                          persistent_workers=(num_workers > 0))
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=True,
                          persistent_workers=(num_workers > 0))
print(f"Loaders ready (num_workers={num_workers}).")

## 🧠 Novel model — FSCG-UNet (Hybrid CNN–ViT + Saliency-Consensus Gates)

In [ ]:
# ------------------------------------------------------------------
# 1. Attention gate (skip fusion -- reference hybrid design)
# ------------------------------------------------------------------
class AttentionGate(nn.Module):
    def __init__(self, g_channels, x_channels, inter_channels):
        super().__init__()
        self.w_g = nn.Conv2d(g_channels, inter_channels, kernel_size=1)
        self.w_x = nn.Conv2d(x_channels, inter_channels, kernel_size=1)
        self.psi = nn.Conv2d(inter_channels, 1, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, g, x):
        g1 = self.w_g(g)
        x1 = self.w_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        psi = self.sigmoid(psi)
        return x * psi


# ------------------------------------------------------------------
# 2. ASPP (multi-scale context)
# ------------------------------------------------------------------
class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels=512, rates=(1, 6, 12, 18)):
        super().__init__()
        self.aspp1 = nn.Conv2d(in_channels, out_channels, 1)
        self.aspp2 = nn.Conv2d(in_channels, out_channels, 3, padding=rates[1], dilation=rates[1])
        self.aspp3 = nn.Conv2d(in_channels, out_channels, 3, padding=rates[2], dilation=rates[2])
        self.aspp4 = nn.Conv2d(in_channels, out_channels, 3, padding=rates[3], dilation=rates[3])
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1),
            nn.ReLU(inplace=True))
        self.out_conv = nn.Conv2d(out_channels * 5, out_channels, 1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        f1, f2, f3, f4 = self.aspp1(x), self.aspp2(x), self.aspp3(x), self.aspp4(x)
        f5 = self.global_pool(x)
        f5 = F.interpolate(f5, size=x.shape[2:], mode="bilinear", align_corners=False)
        out = torch.cat([f1, f2, f3, f4, f5], dim=1)
        out = self.out_conv(out)
        out = self.bn(out)
        return self.relu(out)


# ------------------------------------------------------------------
# 3. Transformer block (ViT bridge)
# ------------------------------------------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=8, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x


# ------------------------------------------------------------------
# 4. Hybrid encoder: ConvNeXt-Tiny features + ViT token bridge
# ------------------------------------------------------------------
class HybridEncoder(nn.Module):
    def __init__(self, img_size=512, patch_size=4, dim=384, depth=8,
                 backbone_name="convnext_tiny", pretrained=True):
        super().__init__()
        self.backbone_name = backbone_name
        backbone = timm.create_model(
            backbone_name, pretrained=pretrained,
            features_only=True, out_indices=(0, 1, 2, 3))

        if "convnext" in backbone_name:
            self.patch_in_ch = 768
            self.skip_channels = [96, 192, 384, 768]
        elif "resnet" in backbone_name:
            self.patch_in_ch = 512
            self.skip_channels = [64, 256, 512, 1024]
        elif "efficientnet" in backbone_name:
            self.patch_in_ch = 320
            self.skip_channels = [24, 48, 112, 320]
        else:
            feat_info = backbone.feature_info
            self.patch_in_ch = feat_info.channels()[-1]
            self.skip_channels = list(feat_info.channels())
        self.backbone = backbone

        # deepest feature map size at stride 32
        low_res = max(1, img_size // 32)
        self.patch_size = max(1, min(patch_size, max(1, low_res // 2)))
        self.patch_embed = nn.Conv2d(self.patch_in_ch, dim,
                                     kernel_size=self.patch_size, stride=self.patch_size)
        max_patches = max(1, (low_res // self.patch_size) ** 2)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_patches, dim))
        self.blocks = nn.ModuleList([TransformerBlock(dim) for _ in range(depth)])

    def forward(self, x):
        features = self.backbone(x)
        x1, x2, x3, x4 = features[0], features[1], features[2], features[3]
        patch_feat = self.patch_embed(x4)
        B, C, h, w = patch_feat.shape
        num_patches = h * w
        pos = self.pos_embedding[:, :num_patches, :]
        tokens = patch_feat.flatten(2).transpose(1, 2) + pos
        for blk in self.blocks:
            tokens = blk(tokens)
        return tokens, x1, x2, x3, x4, h, w


# ------------------------------------------------------------------
# 5. Spatial Saliency-Consensus Gate (novel in-model gate)
# ------------------------------------------------------------------
class SpatialSaliencyGate(nn.Module):
    """Produce a spatial saliency mask from fused features and gate them.
    Returns (gated_features, saliency_map). The saliency map is exposed to the
    server for consensus scoring in Part 3."""
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, 1, 1))

    def forward(self, x):
        logit = self.conv(x)
        mask = torch.sigmoid(logit)
        return x * mask, mask


# ------------------------------------------------------------------
# 6. Decoder block: Upsample + Attention-gated skip + SSCG
# ------------------------------------------------------------------
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.1):
        super().__init__()
        self.dropout = dropout
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True))
        self.att_gate = AttentionGate(out_channels, skip_channels,
                                      max(8, out_channels // 2)) if skip_channels > 0 else None
        total_in = out_channels + skip_channels if skip_channels > 0 else out_channels
        self.conv = nn.Sequential(
            nn.Conv2d(total_in, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Dropout2d(self.dropout if self.training else 0.0))
        self.sal_gate = SpatialSaliencyGate(out_channels)

    def forward(self, x, skip=None):
        x = self.up(x)
        total_in = x.shape[1] + skip.shape[1] if skip is not None else x.shape[1]
        if skip is not None and self.att_gate is not None:
            skip = F.interpolate(skip, size=x.shape[2:], mode="bilinear", align_corners=False)
            skip = self.att_gate(x, skip)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        x, sal = self.sal_gate(x)
        return x, sal


# ------------------------------------------------------------------
# 7. FSCG-UNet
# ------------------------------------------------------------------
class FSCGUNet(nn.Module):
    def __init__(self, num_classes=10, dim=384, depth=8, img_size=512,
                 backbone_name="convnext_tiny", pretrained=True):
        super().__init__()
        self.img_size = img_size
        self.encoder = HybridEncoder(img_size=img_size, patch_size=4, dim=dim,
                                     depth=depth, backbone_name=backbone_name,
                                     pretrained=pretrained)
        self.skip_channels = self.encoder.skip_channels
        self.aspp = ASPP(dim, dim)
        self.dec1 = DecoderBlock(dim,         self.skip_channels[3], 256)
        self.dec2 = DecoderBlock(256,         self.skip_channels[2], 128)
        self.dec3 = DecoderBlock(128,         self.skip_channels[1],  64)
        self.dec4 = DecoderBlock(64,          self.skip_channels[0],  32)
        self.dec5 = DecoderBlock(32,          0,                      32)
        self.head = nn.Conv2d(32, num_classes, 1)
        self.aux_head2 = nn.Conv2d(128, num_classes, 1)
        self.aux_head3 = nn.Conv2d(64,  num_classes, 1)

    def forward(self, x, return_sal=True):
        tokens, x1, x2, x3, x4, H, W = self.encoder(x)
        B, N, C = tokens.shape
        feat = tokens.transpose(1, 2).reshape(B, C, H, W)
        feat = self.aspp(feat)

        feat1, _   = self.dec1(feat, x4)
        feat2, _   = self.dec2(feat1, x3)
        feat3, _   = self.dec3(feat2, x2)
        feat4, _   = self.dec4(feat3, x1)
        feat5, sal = self.dec5(feat4, None)

        main_out = self.head(feat5)
        main_out = F.interpolate(main_out, size=(self.img_size, self.img_size),
                                 mode="bilinear", align_corners=False)
        aux2 = self.aux_head2(feat2)
        aux2 = F.interpolate(aux2, size=(self.img_size, self.img_size),
                             mode="bilinear", align_corners=False)
        aux3 = self.aux_head3(feat3)
        aux3 = F.interpolate(aux3, size=(self.img_size, self.img_size),
                             mode="bilinear", align_corners=False)
        if return_sal:
            return main_out, aux2, aux3, sal
        return main_out

In [ ]:
import os as _os
PRETRAINED = not SMOKE_TEST
model_spec = {
    "arch": "FSCGUNet",
    "num_classes": NUM_CLASSES,
    "dim": int(os.environ.get("FSCG_DIM", "384")),
    "depth": int(os.environ.get("FSCG_DEPTH", "8")),
    "img_size": IMG_SIZE,
    "backbone": "convnext_tiny",
    "pretrained": PRETRAINED,
}

model = FSCGUNet(num_classes=NUM_CLASSES, dim=model_spec["dim"],
                 depth=model_spec["depth"], img_size=model_spec["img_size"],
                 backbone_name=model_spec["backbone"], pretrained=model_spec["pretrained"])
model = model.to(DEVICE)

# ---- freeze early backbone stages to regularize on small data ----
FREEZE_UNTIL = os.environ.get("FSCG_FREEZE", "stage0").lower()
_frozen = 0
for name, p in model.named_parameters():
    _is_backbone = name.startswith("encoder.backbone.")
    if FREEZE_UNTIL == "stem" and name.startswith("encoder.backbone.stem"):
        p.requires_grad = False; _frozen += p.numel()
    elif FREEZE_UNTIL == "stage0" and _is_backbone and (".stages.0." in name or ".stem." in name or ".patch_embed." in name):
        p.requires_grad = False; _frozen += p.numel()
    elif FREEZE_UNTIL == "all" and _is_backbone:
        p.requires_grad = False; _frozen += p.numel()
    elif FREEZE_UNTIL == "none":
        pass
print(f"Frozen {_frozen/1e6:.2f}M backbone params (FREEZE_UNTIL={FREEZE_UNTIL}).")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FSCG-UNet trainable params: {n_params:,} ({n_params/1e6:.2f} M)")

# sanity forward
model.eval()
with torch.no_grad():
    y = model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE), return_sal=True)
    print("main:", tuple(y[0].shape), " aux2:", tuple(y[1].shape), " aux3:", tuple(y[2].shape),
          " sal:", tuple(y[3].shape))
    assert y[0].shape[-2:] == (IMG_SIZE, IMG_SIZE) and y[0].shape[1] == NUM_CLASSES
print("FSCG-UNet forward OK.")
if os.environ.get("FSCG_COMPILE", "0") == "1":
    try:
        model = torch.compile(model, mode="default")
        print("torch.compile ENABLED (mode=default).")
    except Exception as _ce:
        print("torch.compile failed -> running uncompiled:", _ce)

## 🎯 Losses (composite: weighted CE + Dice + Focal + Lovász + deep supervision)

In [ ]:
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum().float()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def _mean(l, empty=0.0):
    l = iter(l)
    try:
        acc = next(l)
    except StopIteration:
        return empty
    n = 1
    for v in l:
        acc += v
        n += 1
    return acc / n


def lovasz_softmax_flat(probas, labels, classes="present"):
    """probas: [N, C]; labels: [N]. Sub-sampled, fast multi-class Lovasz."""
    if probas.numel() == 0:
        return probas * 0.0
    C = probas.size(1)
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if classes == "present" and fg.sum() == 0:
            continue
        class_pred = probas[:, c]
        errors = (fg - class_pred).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg_sorted)))
    return _mean(losses)


def lovasz_softmax(pred, target, classes="present", max_pixels=65536):
    """Sub-sample pixels to keep Lovasz fast on high-res masks."""
    probas = F.softmax(pred, dim=1)
    B, C, H, W = probas.shape
    n_pix = H * W
    if n_pix > max_pixels:
        idx = torch.randperm(n_pix, device=probas.device)[:max_pixels]
        probas = probas.reshape(B, C, -1)[:, :, idx].permute(0, 2, 1).reshape(-1, C)
        target = target.reshape(B, -1)[:, idx].reshape(-1)
    else:
        probas = probas.reshape(-1, C)
        target = target.reshape(-1)
    return lovasz_softmax_flat(probas, target, classes=classes)


class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.softmax(pred, dim=1)
        onehot = torch.nn.functional.one_hot(target, self.num_classes).permute(0, 3, 1, 2).float()
        inter = (pred * onehot).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + onehot.sum(dim=(2, 3))
        dice = (2 * inter + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=3.0, weight=None, ignore_index=-100):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, pred, target):
        ce = F.cross_entropy(pred, target, weight=self.weight,
                             reduction="none", ignore_index=self.ignore_index)
        pt = torch.exp(-ce)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()


class AdvancedLoss(nn.Module):
    def __init__(self, num_classes=10, class_weights=None, device=DEVICE):
        super().__init__()
        if class_weights is not None:
            class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(num_classes)
        self.focal = FocalLoss(alpha=0.75, gamma=3.0, weight=class_weights)

    def forward(self, main_out, aux2, aux3, target):
        main_ce = self.ce(main_out, target)
        main_dice = self.dice(main_out, target)
        main_focal = self.focal(main_out, target)
        main_lovasz = lovasz_softmax(main_out, target)
        loss_main = 0.4 * main_ce + 0.35 * main_dice + 0.15 * main_focal + 0.4 * main_lovasz
        aux2_loss = 0.15 * (self.ce(aux2, target) + self.dice(aux2, target))
        aux3_loss = 0.10 * (self.ce(aux3, target) + self.dice(aux3, target))
        return loss_main + aux2_loss + aux3_loss


def val_loss_fn(pred, target, criterion):
    return 0.5 * criterion.ce(pred, target) + 0.5 * criterion.dice(pred, target)

## 📊 Metrics (GPU-vectorized confusion matrices + TTA)

In [ ]:
@torch.no_grad()
def compute_miou(model, loader, num_classes=NUM_CLASSES, use_tta=False):
    """GPU confusion-matrix mIoU (Optimization 2) + pixel accuracy + per-class IoU."""
    model.eval()
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=DEVICE)
    pix_correct = 0
    pix_total = 0
    for imgs, msks, _ in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        msks = msks.to(DEVICE, non_blocking=True)
        if use_tta:
            preds = tta_predict(model, imgs).argmax(dim=1)
        else:
            preds = torch.argmax(model(imgs)[0], dim=1)
        valid = (msks >= 0) & (msks < num_classes)
        gts = msks[valid]
        prs = preds[valid]
        cm += torch.bincount(num_classes * gts + prs, minlength=num_classes ** 2).reshape(
            num_classes, num_classes)
        pix_correct += (preds == msks).sum().item()
        pix_total += msks.numel()
    inter = torch.diag(cm).float()
    union = cm.sum(dim=1).float() + cm.sum(dim=0).float() - inter
    per_class = torch.where(union > 0, inter / torch.clamp(union, min=1e-8),
                            torch.zeros_like(inter))
    valid = union > 0
    miou = per_class[valid].mean().item() if valid.any() else 0.0
    return miou, {c: float(per_class[c].item()) for c in range(num_classes)}, \
           pix_correct / max(1, pix_total)


def compute_per_class_stats(preds, targets, num_classes):
    preds = preds.argmax(dim=1)
    stats = []
    for cls in range(num_classes):
        p = (preds == cls)
        t = (targets == cls)
        stats.append([(p & t).sum().item(), p.sum().item(), t.sum().item()])
    return stats


def dice_score(pred, target, num_classes=10, smooth=1e-6):
    pred = torch.argmax(pred, dim=1)
    dice = 0.0
    for cls in range(num_classes):
        pc = (pred == cls).float()
        tc = (target == cls).float()
        dice += (2 * (pc * tc).sum() + smooth) / (pc.sum() + tc.sum() + smooth)
    return dice / num_classes


def iou_score(pred, target, num_classes=10, smooth=1e-6):
    pred = torch.argmax(pred, dim=1)
    iou = 0.0
    for cls in range(num_classes):
        pc = (pred == cls)
        tc = (target == cls)
        ic = (pc & tc).float().sum()
        uc = (pc | tc).float().sum()
        iou += (ic + smooth) / (uc + smooth)
    return iou / num_classes


def pixel_accuracy(pred, target):
    pred = torch.argmax(pred, dim=1)
    return (pred == target).float().mean()


@torch.no_grad()
def tta_predict(model, images):
    """Test-time augmentation: original + hflip + vflip + both."""
    preds = []
    main, _, _, _ = model(images)
    preds.append(main)
    h = torch.flip(images, dims=[3]); main_h, _, _, _ = model(h)
    preds.append(torch.flip(main_h, dims=[3]))
    v = torch.flip(images, dims=[2]); main_v, _, _, _ = model(v)
    preds.append(torch.flip(main_v, dims=[2]))
    hv = torch.flip(images, dims=[2, 3]); main_hv, _, _, _ = model(hv)
    preds.append(torch.flip(main_hv, dims=[2, 3]))
    return torch.mean(torch.stack([torch.softmax(p, dim=1) for p in preds]), dim=0)

## ⚙️ Optimizer / scheduler / AMP / early stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=12, path="best_model.pth", trace_func=print):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.trace_func = trace_func

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + 1e-5:
            self.counter += 1
            self.trace_func(f"EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [ ]:
LR = float(os.environ.get("FSCG_LR", "3e-5"))
WEIGHT_DECAY = 1e-3
GRAD_CLIP   = 0.5
PATIENCE    = 12

criterion = AdvancedLoss(num_classes=NUM_CLASSES, class_weights=class_weights,
                         device=DEVICE).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7)

scaler = torch.amp.GradScaler("cuda", enabled=HAS_CUDA) if HAS_CUDA else None
early_stopping = EarlyStopping(patience=PATIENCE,
                               path=os.path.join(CKPT_DIR, "best_fscg_unet_loss.pth"))

print(f"EPOCHS={EPOCHS} LR={LR} clip={GRAD_CLIP} patience={PATIENCE} "
      f"vallog={VAL_LOG_N} scaler={'AMP' if scaler else 'none'}")

## 🔁 Training loop (AMP + scheduling + GPU metrics)

In [ ]:
def train_one_epoch(model, loader, epoch):
    model.train()
    total_loss = total_dice = total_iou = total_acc = 0.0
    n = 0
    loop = tqdm(loader, desc=f"Epoch {epoch+1} train")
    for imgs, msks, _ in loop:
        imgs = imgs.to(DEVICE, non_blocking=True)
        msks = msks.long().to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=HAS_CUDA):
            main_out, aux2, aux3, _ = model(imgs)
            loss = criterion(main_out, aux2, aux3, msks)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        with torch.no_grad():
            d, i, a = dice_score(main_out, msks), iou_score(main_out, msks), pixel_accuracy(main_out, msks)
        bs = imgs.size(0)
        total_loss += loss.item() * bs
        total_dice += d.item() * bs
        total_iou  += i.item() * bs
        total_acc  += a.item() * bs
        n += bs
        loop.set_postfix(loss=loss.item(), dice=d.item(), iou=i.item())
    return total_loss / max(1, n), total_dice / max(1, n), total_iou / max(1, n), total_acc / max(1, n)


def validate(model, loader, subset_n=None):
    model.eval()
    total_loss = total_dice = total_iou = total_acc = 0.0
    n = 0
    total_stats = [[0, 0, 0] for _ in range(NUM_CLASSES)]
    shown = 0
    with torch.no_grad():
        loop = tqdm(loader, desc="Val")
        for imgs, msks, _ in loop:
            imgs = imgs.to(DEVICE, non_blocking=True)
            msks = msks.long().to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=HAS_CUDA):
                outputs = tta_predict(model, imgs)
            loss = val_loss_fn(outputs, msks, criterion)
            d, i, a = dice_score(outputs, msks), iou_score(outputs, msks), pixel_accuracy(outputs, msks)
            stats = compute_per_class_stats(outputs, msks, NUM_CLASSES)
            for k in range(NUM_CLASSES):
                for j in range(3):
                    total_stats[k][j] += stats[k][j]
            bs = imgs.size(0)
            total_loss += loss.item() * bs
            total_dice += d.item() * bs
            total_iou  += i.item() * bs
            total_acc  += a.item() * bs
            n += bs
            shown += 1
            if subset_n and shown >= subset_n:
                break
    dice_cls = [(2 * s[0] + 1e-6) / (s[1] + s[2] + 1e-6) for s in total_stats]
    iou_cls  = [(s[0] + 1e-6) / (s[1] + s[2] - s[0] + 1e-6) for s in total_stats]
    return (total_loss / max(1, n), total_dice / max(1, n), total_iou / max(1, n),
            total_acc / max(1, n), dice_cls, iou_cls)


print("train_one_epoch + validate defined.")

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": [],
           "train_iou": [], "val_iou": [], "train_acc": [], "val_acc": [],
           "per_class_dice": [], "per_class_iou": [], "lr": []}

best_dice, best_iou = 0.0, 0.0
t0 = time.time()
last_val = (0.0, 0.0, 0.0, 0.0, [0.0]*NUM_CLASSES, [0.0]*NUM_CLASSES)
for epoch in range(EPOCHS):
    print(f"\n{'='*60}\nEpoch {epoch+1}/{EPOCHS}")
    if (time.time() - t0) / 3600 >= MAX_HOURS:
        print(f"  [TIME-BUDGET] {MAX_HOURS:.1f}h reached after {epoch}/{EPOCHS} epochs; stopping training (best checkpoint kept).")
        break
    tr_loss, tr_dice, tr_iou, tr_acc = train_one_epoch(model, train_loader, epoch)
    if (epoch % VAL_EVERY) == 0 or epoch == EPOCHS - 1:
        val_loss, val_dice, val_iou, val_acc, dice_cls, iou_cls = validate(model, val_loader,
                                                                          subset_n=VAL_LOG_N)
        last_val = (val_loss, val_dice, val_iou, val_acc, dice_cls, iou_cls)
    else:
        val_loss, val_dice, val_iou, val_acc, dice_cls, iou_cls = last_val

    scheduler.step()
    history["lr"].append(optimizer.param_groups[0]["lr"])

    history["train_loss"].append(tr_loss); history["val_loss"].append(val_loss)
    history["train_dice"].append(tr_dice); history["val_dice"].append(val_dice)
    history["train_iou"].append(tr_iou);   history["val_iou"].append(val_iou)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(val_acc)
    history["per_class_dice"].append(dice_cls)
    history["per_class_iou"].append(iou_cls)

    print(f"Train -> Acc {tr_acc:.4f} | Loss {tr_loss:.4f} | Dice {tr_dice:.4f} | IoU {tr_iou:.4f}")
    print(f"Val   -> Acc {val_acc:.4f} | Loss {val_loss:.4f} | Dice {val_dice:.4f} | IoU {val_iou:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best_fscg_unet_dice.pth"))
        print("-> Best (val Dice) saved.")
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best_fscg_unet_iou.pth"))
        print("-> Best (val IoU) saved.")

    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("\nEarly stopping triggered.")
        break

    _elp = time.time() - t0
    _per = _elp / (epoch + 1)
    print(f"  [epoch {epoch+1}/{EPOCHS}] {_elp:.0f}s elapsed | avg {_per:.0f}s/epoch | ETA {_per*(EPOCHS-(epoch+1))/3600:.1f}h")
    if _elp / 3600 >= MAX_HOURS and epoch < EPOCHS - 1:
        print(f"  [TIME-BUDGET] {MAX_HOURS:.1f}h reached after {epoch+1} epochs; stopping (best checkpoint kept).")
        break

print(f"\nTraining done in {(time.time()-t0)/60:.1f} min | best val Dice {best_dice:.4f} | best val IoU {best_iou:.4f}")

## 📈 Training curves

In [ ]:
epochs_ax = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (k_tr, k_va, title) in zip(axes, [("train_loss", "val_loss", "Loss"),
                                           ("train_dice", "val_dice", "Dice"),
                                           ("train_iou", "val_iou", "IoU")]):
    ax.plot(epochs_ax, history[k_tr], label="train")
    ax.plot(epochs_ax, history[k_va], label="val")
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3); ax.set_xlabel("epoch")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "11_central_train_curves.png"), dpi=140, bbox_inches="tight")
plt.show()
print("11_central_train_curves.png saved.")

## ✅ Final evaluation on the real TEST split (TTA)

In [ ]:
# load best checkpoint
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "best_fscg_unet_iou.pth"),
                                 map_location=DEVICE))

test_miou, test_per_class, test_pixel_acc = compute_miou(model, test_loader, use_tta=True)
print("\n===== CENTRALIZED FSCG-UNet TEST (real GT, TTA) =====")
print(f"mIoU      : {test_miou:.4f}")
print(f"Pixel Acc : {test_pixel_acc:.4f}")
for c in range(NUM_CLASSES):
    print(f"  {c:2d} {CLASS_NAMES[c]:<22} IoU {test_per_class[c]:.4f}")

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(NUM_CLASSES)
ax.bar(x, [test_per_class[c] for c in range(NUM_CLASSES)], color="tab:blue")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_ylabel("IoU"); ax.set_title("FSCG-UNet per-class IoU (test)")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "12_central_perclass_iou.png"), dpi=140, bbox_inches="tight")
plt.show()
print("12_central_perclass_iou.png saved.")

## 🎚️ Learning-rate schedule

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(history["lr"]) + 1), history["lr"], "o-", color="tab:green")
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning rate")
ax.set_title("CosineAnnealingWarmRestarts schedule")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "12b_central_lr_schedule.png"), dpi=140, bbox_inches="tight")
plt.show()
print("12b_central_lr_schedule.png saved.")

## 🌡️ Epoch-wise per-class IoU evolution (validation subset)

In [ ]:
iou_hist = np.array(history["per_class_iou"])       # [epoch, class]
fig, ax = plt.subplots(figsize=(12, max(3.5, 0.4 * iou_hist.shape[0])))
im = ax.imshow(iou_hist, aspect="auto", cmap="YlGnBu", vmin=0.0, vmax=1.0)
ax.set_yticks(range(iou_hist.shape[0]))
ax.set_yticklabels([f"E{e+1}" for e in range(iou_hist.shape[0])], fontsize=8)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_title("Per-class validation IoU across epochs")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "12c_central_iou_epochs.png"), dpi=140, bbox_inches="tight")
plt.show()
print("12c_central_iou_epochs.png saved.")

## 🔢 Confusion matrix (test, TTA)

In [ ]:
@torch.no_grad()
def test_confusion_matrix(model, loader, use_tta=True):
    model.eval()
    cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=DEVICE)
    for img_b, msk_b, _ in loader:
        img_b = img_b.to(DEVICE, non_blocking=True)
        logits = tta_predict(model, img_b) if use_tta else model(img_b)[0]
        preds = logits.argmax(dim=1)
        msk_b = msk_b.to(DEVICE, non_blocking=True)
        flat_p, flat_m = preds.view(-1), msk_b.view(-1)
        valid = flat_m < NUM_CLASSES
        cm += torch.bincount(flat_m[valid] * NUM_CLASSES + flat_p[valid],
                             minlength=NUM_CLASSES * NUM_CLASSES).view(NUM_CLASSES, NUM_CLASSES)
    return cm.cpu().numpy()

cm_test = test_confusion_matrix(model, test_loader, use_tta=True)
cm_norm = cm_test / np.clip(cm_test.sum(axis=1, keepdims=True), 1e-8, None)

fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1.0)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="black")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("FSCG-UNet normalized confusion matrix (test, TTA)")
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "12d_central_confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()
print("12d_central_confusion_matrix.png saved.")
np.save(os.path.join(OUTPUT_DIR, "central_confusion_matrix.npy"), cm_test)

## 📋 Per-class metrics table (IoU + Dice) figure + CSV

In [ ]:
tp = np.diag(cm_test).astype(float)
fp = cm_test.sum(axis=0) - tp
fn = cm_test.sum(axis=1) - tp
iou_test = tp / np.clip(tp + fp + fn, 1e-8, None)
dice_test = 2 * tp / np.clip(2 * tp + fp + fn, 1e-8, None)

result_table = pd.DataFrame({
    "Class": [f"{c} {CLASS_NAMES[c]}" for c in range(NUM_CLASSES)],
    "IoU": iou_test.round(4),
    "Dice": dice_test.round(4),
    "Exact-match acc": (tp / np.clip(cm_test.sum(axis=1), 1e-8, None)).round(4),
})
result_table.to_csv(os.path.join(OUTPUT_DIR, "tables", "p2_central_per_class_metrics.csv"), index=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")
tbl = ax.table(cellText=result_table.round(4).values, colLabels=result_table.columns,
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.1, 1.6)
ax.set_title(f"FSCG-UNet centralized -- test per-class metrics  |  mIoU {test_miou:.4f} | pixel-acc {test_pixel_acc:.4f}",
             fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "12e_central_metrics_table.png"), dpi=150, bbox_inches="tight")
plt.show()
print("12e_central_metrics_table.png saved + tables/p2_central_per_class_metrics.csv")

## 🖼️ Visualize predictions

In [ ]:
def visualize_samples(model, loader, n=4):
    model.eval()
    imgs, msks, names = next(iter(loader))
    imgs = imgs.to(DEVICE)
    with torch.no_grad():
        outputs = tta_predict(model, imgs)
    preds = outputs.argmax(dim=1).cpu().numpy()
    imgs_np = imgs.cpu().permute(0, 2, 3, 1).numpy()
    msks_np = msks.numpy()
    denorm = lambda t: np.clip(t * np.array([0.229, 0.224, 0.225]) +
                               np.array([0.485, 0.456, 0.406]), 0, 1)
    n = min(n, len(imgs_np))
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[None, :]
    for i in range(n):
        axes[i, 0].imshow(denorm(imgs_np[i])); axes[i, 0].set_title(f"Image - {names[i]}"); axes[i, 0].axis("off")
        axes[i, 1].imshow(msks_np[i], cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 1].set_title("GT"); axes[i, 1].axis("off")
        axes[i, 2].imshow(preds[i], cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 2].set_title("Pred"); axes[i, 2].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "13_central_predictions.png"), dpi=140, bbox_inches="tight")
    plt.show()

visualize_samples(model, val_loader, n=min(3, BATCH_SIZE))
print("13_central_predictions.png saved.")

## 🌊 SSCG saliency visualisation (decoder step-5 map from FSCG-UNet)

In [ ]:
def visualize_saliency(model, loader, n=4):
    model.eval()
    imgs, msks, names = next(iter(loader))
    imgs = imgs.to(DEVICE)
    with torch.no_grad():
        main_out, _, _, sal = model(imgs)          # sal shape B,1,Hf,Wf (dec5 res)
        preds = main_out.argmax(dim=1)
    sal_up = F.interpolate(sal, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    sal_np = sal_up.squeeze(1).cpu().numpy()
    preds_np = preds.cpu().numpy()
    msks_np = msks.numpy()
    imgs_np = imgs.cpu().permute(0, 2, 3, 1).numpy()
    denorm = lambda t: np.clip(t * np.array([0.229, 0.224, 0.225]) +
                               np.array([0.485, 0.456, 0.406]), 0, 1)
    n = min(n, len(imgs_np))
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[None, :]
    for i in range(n):
        axes[i, 0].imshow(denorm(imgs_np[i])); axes[i, 0].set_title(f"Image - {names[i]}"); axes[i, 0].axis("off")
        axes[i, 1].imshow(msks_np[i], cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 1].set_title("GT"); axes[i, 1].axis("off")
        axes[i, 2].imshow(preds_np[i], cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 2].set_title("Pred"); axes[i, 2].axis("off")
        axes[i, 3].imshow(denorm(imgs_np[i]), alpha=0.7)
        axes[i, 3].imshow(sal_np[i], cmap="jet", alpha=0.6)
        axes[i, 3].set_title("SSCG saliency"); axes[i, 3].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "14_central_saliency.png"), dpi=140, bbox_inches="tight")
    plt.show()
    print("14_central_saliency.png saved.")

visualize_saliency(model, val_loader, n=min(3, BATCH_SIZE))

## 📊 Hyperparameter tracker + parameter budget (Part 2)

In [ ]:
total_p   = sum(p.numel() for p in model.parameters())
train_p   = sum(p.numel() for p in model.parameters() if p.requires_grad)
per_module = {k: int(sum(p.numel() for p in m.parameters() if p.requires_grad))
              for k, m in model.named_children()}

hp = {
    "part": 2,
    "reference": "hybrid-cnn-vit-attention-u-net (notebook) strategy",
    "model": "FSCG-UNet (Hybrid CNN+ViT, deep-supervised)",
    "backbone": model_spec.get("backbone"),
    "vit_dim": model_spec.get("dim"),
    "vit_depth": model_spec.get("depth"),
    "img_size": IMG_SIZE,
    "num_classes": NUM_CLASSES,
    "total_params": int(total_p),
    "trainable_params": int(train_p),
    "per_module_trainable": {k: int(v) for k, v in per_module.items()},
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "decoder_dropout": 0.1,
    "freeze_until": os.environ.get("FSCG_FREEZE", "stage0"),
    "train_aug": "normal aug on originals + strong aug on oversampled copies",
    "train_samples": len(train_dataset),
    "optimizer": type(optimizer).__name__,
    "scheduler": type(scheduler).__name__ + " (T_0=10, T_mult=2, eta_min=1e-7)",
    "epochs_run": len(history["train_loss"]),
    "epochs_cfg": EPOCHS,
    "grad_clip": GRAD_CLIP,
    "patience": PATIENCE,
    "val_log_n": VAL_LOG_N,
    "early_stopped": early_stopping.early_stop,
    "best_val_dice": float(best_dice),
    "best_val_iou": float(best_iou),
    "test_miou": float(test_miou),
    "test_pixel_acc": float(test_pixel_acc),
    "loss": "AdvancedLoss = wCE + Dice + Focal + Lovasz + deep-supervision aux",
    "amp": "autocast + GradScaler",
    "tta": "4-view flip averaging at eval (never on labels)",
    "class_weights": ([round(float(w), 4) for w in class_weights]
                      if class_weights is not None else "uniform"),
    "seed": SEED,
}
hp_flat = pd.DataFrame({"hyperparameter": list(hp), "value": [str(v) for v in hp.values()]})
with open(os.path.join(OUTPUT_DIR, "tables", "p2_hyperparams.json"), "w") as f:
    json.dump(hp, f, indent=2, default=str)
hp_flat.to_csv(os.path.join(OUTPUT_DIR, "tables", "p2_hyperparams.csv"), index=False)
print("Part-2 hyperparameters (saved to tables/p2_hyperparams.{json,csv}):")
for k, v in hp.items():
    print(f"  {k:<28} {str(v)[:90]}")
print(f"  {'params/module':<28} " + " ".join(f"{k}={v:,}" for k, v in per_module.items()))

## 💾 Save Part-2 outputs (reused by Part 3)

In [ ]:
# Keep the best checkpoint so Part 3 could initialise a global model.
torch.save(model.state_dict(), os.path.join(CKPT_DIR, "fscg_unet_best.pth"))

result = {
    "model_id": "centralized_fscg_unet",
    "display_name": "FSCG-UNet (centralized baseline)",
    "params_m": round(n_params / 1e6, 2),
    "epochs": len(history["train_loss"]),
    "best_val_dice": best_dice,
    "best_val_iou": best_iou,
    "test_miou": test_miou,
    "test_pixel_acc": test_pixel_acc,
    "test_per_class_iou": {str(k): round(v, 4) for k, v in test_per_class.items()},
    "model_spec": model_spec,
    "history": history,
    "config": {"img_size": IMG_SIZE, "batch": BATCH_SIZE, "lr": LR,
               "epochs": EPOCHS, "loss": "AdvancedLoss (CE+Dice+Focal+Lovasz+aux)"},
}
with open(os.path.join(OUTPUT_DIR, "part2_centralized_fscg_unet.json"), "w") as f:
    json.dump(result, f, indent=2, default=float)
with open(os.path.join(OUTPUT_DIR, "model_spec.json"), "w") as f:
    json.dump(model_spec, f, indent=2)
print("Saved part2_centralized_fscg_unet.json + model_spec.json")

txt = [
    "Fed-SALGATE Part 2 - Centralized FSCG-UNet summary",
    "=" * 50,
    f"Test mIoU      : {test_miou:.4f}",
    f"Test pixel acc : {test_pixel_acc:.4f}",
    f"Best val dice  : {best_dice:.4f}",
    f"Best val iou   : {best_iou:.4f}",
    f"Params (M)     : {n_params/1e6:.2f}",
]
with open(os.path.join(OUTPUT_DIR, "PART2_SUMMARY.txt"), "w") as f:
    f.write("\n".join(txt))
print("\n".join(txt))

# ZIP the (small) outputs so they can be transferred from any device
import zipfile
_zip_path = os.path.join(WORKING_DIR, "part2_outputs.zip")
with zipfile.ZipFile(_zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            if f.endswith(".pth"):
                continue
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, WORKING_DIR))
print(f"Zipped Part-2 outputs -> {_zip_path}  ({os.path.getsize(_zip_path)/1e6:.2f} MB)")
print("\nDONE -- Part 2 finished. Attach this notebook's Output (with Part 1's) to Part 3.")